# Day 4 v2 — Model 12: AITeamVN Full Encoder Fine-tune (all 24 layers)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (568M, 24L, 1024-dim) — unfreeze **all 24 layers** → mean_pooling → price head + aux category head.

**vs NB09 (top-8, MAE pending):** Full encoder gives max capacity; LLRD decay=0.70 keeps bottom layers at lr≈1e-7 to avoid catastrophic forgetting.

| Config | Value | Reason |
|---|---|---|
| keep_top_layers | 24 (all) | Maximum encoder capacity |
| batch / grad_accum | 16 / 2 (eff=32) | Fit full 568M grad in VRAM |
| base_lr | 1.5e-5 | Lower to protect pretrained weights |
| llrd_decay | 0.70 | Bottom layers lr≈1e-7 — near frozen |
| warmup_ratio | 0.10 | Longer warmup for full encoder |
| R-Drop alpha | 0.3 | MSE consistency regularization |
| EMA decay | 0.9999 | ~10K step window |
| epochs | 8 | ≤10h budget on 3090 Ti |

**Target:** MAE < 68k VND

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.bert_finetune_model import BERTFinetuneRunner

MODEL_NAME   = "AITeamVN/Vietnamese_Embedding"
WEIGHT_DIR   = Path("weights")
VAL_PRED_DIR = Path("val_predictions")

# A12 config — full 24 layers, aggressive LLRD
KEEP_TOP     = 24      # all 24 layers unfrozen
BATCH        = 16      # small to fit full 568M grad
GRAD_ACCUM   = 2       # effective batch = 32 (remove if OOM)
BASE_LR      = 1.5e-5  # lower lr for full encoder
WEIGHT_DECAY = 0.01
LLRD_DECAY   = 0.70    # very aggressive: bottom layers lr ≈ 1e-7
EPOCHS       = 8
PATIENCE     = 3
EMA_DECAY    = 0.9999
WARMUP_RATIO = 0.10    # longer warmup for full encoder
R_DROP_ALPHA = 0.3

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Runner

- Tokenize 269K train + 3926 val (~550MB at max_length=256)
- Unfreeze all 24 layers → LLRD: head lr=1.5e-5, layer 23 lr≈1.1e-5, layer 0 lr≈1e-7
- Approx trainable params: ~568M total encoder (all layers trainable)

In [ ]:
runner = BERTFinetuneRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=KEEP_TOP,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
)

## 3. Train

8 epochs, early stopping patience=3. Val MAE on full 3926 samples per epoch.
grad_accum=2 → effective batch=32. Expected: ~60-80 min/epoch on RTX 3090 Ti.

In [ ]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
    grad_accum=GRAD_ACCUM,
)

## 4. Training History

In [ ]:
plot_training_history(history, title="AITeamVN Full Encoder (all 24 layers, LLRD 0.70)")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "aitvn_full.pth"))
print("Saved weights/aitvn_full.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "aitvn_full_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/aitvn_full_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open(VAL_PRED_DIR / "aitvn_full_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/aitvn_full_test.json ({len(test_preds)} samples)")

## 6. Evaluate on 200 Test Samples

In [ ]:
def aitvn_full_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_full_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "aitvn_full.pth"), map_location="cpu", weights_only=False)
print(f"\nCheckpoint keys: {sorted(ckpt.keys())}")
print(f"keep_top_layers={ckpt['keep_top_layers']} | model_name={ckpt['model_name']}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f}")